# Investigate Problem Trade: 2025-01-09

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DATA_DIR = Path("../data/daily")

price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
realized_loss = pd.read_parquet(DATA_DIR / "realized_loss.parquet").rename(columns={"value": "realized_loss"}).set_index("time")

df = price.join(mvrv, how='inner').join(sopr, how='inner').join(sopr_sth, how='inner').join(realized_loss, how='inner')
df['rl_ma30'] = df['realized_loss'].rolling(30).mean()
df['rl_std30'] = df['realized_loss'].rolling(30).std()
df['rl_zscore'] = (df['realized_loss'] - df['rl_ma30']) / df['rl_std30']

# Focus on the problem trade period
trade_period = df[(df.index >= '2025-01-01') & (df.index <= '2025-04-15')].copy()
print(f"Analyzing: {trade_period.index.min().date()} to {trade_period.index.max().date()}")

In [ ]:
# Check entry day conditions
entry_date = '2025-01-09'
entry_data = df.loc[entry_date]

print("ENTRY CONDITIONS ON 2025-01-09")
print("="*50)
print(f"Price: ${entry_data['price']:,.0f}")
print(f"MVRV: {entry_data['mvrv']:.2f}")
print(f"SOPR: {entry_data['sopr']:.4f} {'✅ < 1' if entry_data['sopr'] < 1 else '❌ >= 1'}")
print(f"STH-SOPR: {entry_data['sopr_sth']:.4f} {'✅ < 1' if entry_data['sopr_sth'] < 1 else '❌ >= 1'}")
print(f"RL Z-Score: {entry_data['rl_zscore']:.2f} {'✅ > 0.5' if entry_data['rl_zscore'] > 0.5 else '❌ <= 0.5'}")

print(f"\n⚠️ MVRV at entry: {entry_data['mvrv']:.2f}")
if entry_data['mvrv'] >= 2.0:
    print("   Trail was IMMEDIATELY active!")
else:
    print(f"   Trail activates at MVRV > 2.0 (need +{(2.0/entry_data['mvrv']-1)*100:.0f}% more)")

In [ ]:
# Track the trade day by day
entry_price = 93231
exit_date = '2025-04-07'

trade_df = df[(df.index >= entry_date) & (df.index <= exit_date)].copy()
trade_df['gain_from_entry'] = (trade_df['price'] / entry_price - 1) * 100
trade_df['peak_price'] = trade_df['price'].cummax()
trade_df['trail_stop'] = trade_df['peak_price'] * 0.75
trade_df['trail_active'] = trade_df['mvrv'] >= 2.0

# Find when trail first activated
trail_activated = trade_df[trade_df['trail_active']].index.min()
peak_at_activation = trade_df.loc[trail_activated, 'peak_price'] if pd.notna(trail_activated) else None

print("\nTRADE TIMELINE")
print("="*80)
print(f"Entry: {entry_date} at ${entry_price:,.0f}")
print(f"Peak: ${trade_df['peak_price'].max():,.0f} ({(trade_df['peak_price'].max()/entry_price-1)*100:+.1f}%)")
print(f"Trail activated: {trail_activated.date() if pd.notna(trail_activated) else 'Never'}")
if pd.notna(trail_activated):
    print(f"MVRV at activation: {trade_df.loc[trail_activated, 'mvrv']:.2f}")
    print(f"Peak price at activation: ${peak_at_activation:,.0f}")
print(f"Exit: {exit_date} at ${trade_df.loc[exit_date, 'trail_stop']:,.0f}")

In [ ]:
# Visualize the trade
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, row_heights=[0.5, 0.25, 0.25],
                    subplot_titles=['Price & Trail Stop', 'MVRV', 'Gain from Entry'])

# Price and trail stop
fig.add_trace(go.Scatter(x=trade_df.index, y=trade_df['price'], name='Price', line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=trade_df.index, y=trade_df['trail_stop'], name='Trail Stop (25% from peak)', 
                         line=dict(color='red', dash='dash')), row=1, col=1)
fig.add_hline(y=entry_price, line_dash='dot', line_color='green', row=1, col=1, 
              annotation_text=f'Entry ${entry_price:,}')

# Entry/exit markers
fig.add_trace(go.Scatter(x=[pd.Timestamp(entry_date)], y=[entry_price], mode='markers',
                         name='Entry', marker=dict(color='green', size=15, symbol='triangle-up')), row=1, col=1)
fig.add_trace(go.Scatter(x=[pd.Timestamp(exit_date)], y=[trade_df.loc[exit_date, 'trail_stop']], mode='markers',
                         name='Exit', marker=dict(color='red', size=15, symbol='triangle-down')), row=1, col=1)

# MVRV
fig.add_trace(go.Scatter(x=trade_df.index, y=trade_df['mvrv'], name='MVRV', line=dict(color='purple')), row=2, col=1)
fig.add_hline(y=2.0, line_dash='dash', line_color='red', row=2, col=1, annotation_text='Trail Trigger (2.0)')

# Gain from entry
fig.add_trace(go.Scatter(x=trade_df.index, y=trade_df['gain_from_entry'], name='Gain %', 
                         fill='tozeroy', line=dict(color='green')), row=3, col=1)
fig.add_hline(y=0, line_dash='solid', line_color='black', row=3, col=1)

fig.update_layout(height=800, title_text='Problem Trade Analysis: 2025-01-09 Entry')
fig.show()

In [ ]:
# Key moments table
print("\nKEY MOMENTS")
print("="*90)
print(f"{'Date':<12} {'Price':>10} {'MVRV':>8} {'Gain':>8} {'Peak':>10} {'Trail Stop':>12} {'Status'}")
print("-"*90)

# Show every 7 days + key moments
key_dates = [entry_date]
if pd.notna(trail_activated):
    key_dates.append(str(trail_activated.date()))
key_dates.append(exit_date)

# Add weekly samples
for i in range(0, len(trade_df), 7):
    key_dates.append(str(trade_df.index[i].date()))

key_dates = sorted(set(key_dates))

for date in key_dates:
    if date in trade_df.index.astype(str).values:
        row = trade_df.loc[date]
        status = ""
        if date == entry_date:
            status = "← ENTRY"
        elif date == exit_date:
            status = "← EXIT (trail hit)"
        elif row['mvrv'] >= 2.0 and trade_df.loc[:date, 'mvrv'].lt(2.0).any():
            if str(trail_activated.date()) == date:
                status = "← TRAIL ACTIVATED"
        
        print(f"{date:<12} ${row['price']:>9,.0f} {row['mvrv']:>8.2f} {row['gain_from_entry']:>+7.1f}% "
              f"${row['peak_price']:>9,.0f} ${row['trail_stop']:>11,.0f} {status}")

In [ ]:
# What could have prevented this?
print("\n" + "="*60)
print("ANALYSIS: WHAT WENT WRONG?")
print("="*60)

entry_mvrv = df.loc[entry_date, 'mvrv']
max_gain = (trade_df['price'].max() / entry_price - 1) * 100

print(f"\n1. MVRV at entry: {entry_mvrv:.2f}")
if entry_mvrv > 1.8:
    print(f"   ⚠️ Already elevated! Close to trail trigger.")
    print(f"   → Consider adding MVRV < 1.5 entry filter?")

print(f"\n2. Max gain during trade: {max_gain:+.1f}%")
print(f"   25% trail from peak = {max_gain - 25:.1f}% from entry")
if max_gain < 25:
    print(f"   ⚠️ Never had enough buffer to survive 25% trail!")

print(f"\n3. Entry signal was valid:")
print(f"   SOPR < 1: {df.loc[entry_date, 'sopr']:.4f} ✅")
print(f"   STH-SOPR < 1: {df.loc[entry_date, 'sopr_sth']:.4f} ✅")
print(f"   RL Z > 0.5: {df.loc[entry_date, 'rl_zscore']:.2f} ✅")
print(f"   → Signal was legitimate capitulation, but at elevated MVRV")